# Testing Identity Propagation in Amazon Bedrock AgentCore

This tutorial demonstrates JWT-based identity propagation in AgentCore Runtime using Amazon Cognito.

## Key Learnings

- **ID Token** contains custom attributes (tenant_id, user_role) → Use for multi-tenant apps
- **Access Token** contains only basic identity (sub, username) → Use for simple auth
- JWT propagates: Client → AgentCore Runtime → BedrockAgentCoreContext → Agent

In [1]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands

  error: subprocess-exited-with-error
  
  × Building wheel for strands (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [273 lines of output]
      toml section missing PosixPath('pyproject.toml') does not contain a tool.setuptools_scm section
      running bdist_wheel
      running build
      running build_ext
      
      ==> Configuring:
      $ cmake -S /private/var/folders/0m/yqm62j6n7mlg2p7338lcckv80000gq/T/pip-install-poykcpnw/strands_f3d58442a2084359a07a204c787f9b1d -B /private/var/folders/0m/yqm62j6n7mlg2p7338lcckv80000gq/T/pip-install-poykcpnw/strands_f3d58442a2084359a07a204c787f9b1d/build/temp.macosx-14.3-arm64-cpython-310_Strands -G Ninja -DCMAKE_MAKE_PROGRAM=/private/var/folders/0m/yqm62j6n7mlg2p7338lcckv80000gq/T/pip-build-env-oxmb9li4/overlay/bin/ninja -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX:PATH=/private/var/folders/0m/yqm62j6n7mlg2p7338lcckv80000gq/T/pip-install-poykcpnw/strands_f3d58442a2084359a07a204c787f9b1d/build/lib.macosx-14.3-arm64-c

In [2]:
import boto3
import json
import time
import base64
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
cognito_client = boto3.client('cognito-idp', region_name=region)

print(f"Region: {region}")
print(f"Account ID: {account_id}")

Region: us-west-2
Account ID: 269550163595


## Step 1: Setup Cognito with Custom Attributes

In [3]:
user_pool_name = f"agentcore-id-test-{int(time.time())}"

user_pool_response = cognito_client.create_user_pool(
    PoolName=user_pool_name,
    Policies={'PasswordPolicy': {'MinimumLength': 8}},
    Schema=[
        {'Name': 'email', 'AttributeDataType': 'String', 'Mutable': True, 'Required': True},
        {'Name': 'tenant_id', 'AttributeDataType': 'String', 'Mutable': True},
        {'Name': 'user_role', 'AttributeDataType': 'String', 'Mutable': True}
    ],
    AutoVerifiedAttributes=['email']
)

user_pool_id = user_pool_response['UserPool']['Id']

app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName=f"{user_pool_name}-client",
    GenerateSecret=False,
    ExplicitAuthFlows=['ALLOW_USER_PASSWORD_AUTH', 'ALLOW_REFRESH_TOKEN_AUTH']
)

client_id = app_client_response['UserPoolClient']['ClientId']
discovery_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

cognito_config = {
    'user_pool_id': user_pool_id,
    'client_id': client_id,
    'discovery_url': discovery_url
}

print(f"✓ User Pool: {user_pool_id}")
print(f"✓ Client ID: {client_id}")

✓ User Pool: us-west-2_WS5x9it9p
✓ Client ID: 5gmm9nquhjt9df0rok9fvkdb77


In [4]:
test_users = [
    {'username': 'user-a', 'password': 'TempPass123!', 'email': 'user-a@example.com', 
     'tenant_id': 'tenant-a', 'user_role': 'admin'},
    {'username': 'user-b', 'password': 'TempPass123!', 'email': 'user-b@example.com', 
     'tenant_id': 'tenant-b', 'user_role': 'user'}
]

for user in test_users:
    try:
        cognito_client.admin_create_user(
            UserPoolId=user_pool_id,
            Username=user['username'],
            UserAttributes=[
                {'Name': 'email', 'Value': user['email']},
                {'Name': 'email_verified', 'Value': 'true'},
                {'Name': 'custom:tenant_id', 'Value': user['tenant_id']},
                {'Name': 'custom:user_role', 'Value': user['user_role']}
            ],
            TemporaryPassword=user['password'],
            MessageAction='SUPPRESS'
        )
        cognito_client.admin_set_user_password(
            UserPoolId=user_pool_id,
            Username=user['username'],
            Password=user['password'],
            Permanent=True
        )
        print(f"✓ {user['username']} (Tenant: {user['tenant_id']})")
    except cognito_client.exceptions.UsernameExistsException:
        print(f"↻ {user['username']} already exists")

✓ user-a (Tenant: tenant-a)
✓ user-b (Tenant: tenant-b)


## Step 2: Create Identity-Aware Agent

In [5]:
%%writefile identity_aware_agent.py
from strands import Agent
import json
import base64
from bedrock_agentcore.runtime import BedrockAgentCoreApp, BedrockAgentCoreContext
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

def decode_jwt(token):
    try:
        parts = token.split('.')
        payload = parts[1]
        padding = 4 - (len(payload) % 4)
        if padding != 4:
            payload += '=' * padding
        return json.loads(base64.b64decode(payload))
    except:
        return None

model = BedrockModel(model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0")
agent = Agent(model=model, system_prompt="You're a helpful assistant.")

@app.entrypoint
def identity_aware_agent(payload):
    headers = BedrockAgentCoreContext.get_request_headers() or {}
    auth = headers.get('authorization') or headers.get('Authorization')
    
    user_identity = {'authenticated': False, 'tenant_id': None, 'user_role': None, 'username': None}
    
    if auth:
        token = auth.replace('Bearer ', '').replace('bearer ', '')
        claims = decode_jwt(token)
        if claims:
            user_identity = {
                'authenticated': True,
                'tenant_id': claims.get('custom:tenant_id'),
                'user_role': claims.get('custom:user_role'),
                'username': claims.get('username') or claims.get('cognito:username'),
                'sub': claims.get('sub')
            }
    
    print(f"Identity: {json.dumps(user_identity)}")
    
    response = agent(payload.get("prompt", ""))
    
    return {
        "response": response.message['content'][0]['text'],
        "user_identity": user_identity
    }

if __name__ == "__main__":
    app.run()

Overwriting identity_aware_agent.py


## Step 3: Deploy to AgentCore Runtime

**Important**: We'll try WITHOUT `allowedAudience` first to see if that's the issue.

In [41]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="identity_aware_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="identity_test",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config['discovery_url'],
            "allowedAudience": [cognito_config['client_id']]
        }
    },
    request_header_configuration={  # ← 이게 필요!
        "requestHeaderAllowlist": ["Authorization"]
    }
)

print("✓ Configured WITH Authorization header forwarding")

Entrypoint parsed: file=/Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/identity_aware_agent.py, bedrock_agentcore_name=identity_aware_agent
Memory configured with STM only
Configuring BedrockAgentCore agent: identity_test


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Will create new memory with mode: STM_ONLY
Memory configuration: Short-term memory only
Found existing memory ID from previous launch: identity_test_mem-gx6EYnC8r5
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/.dockerignore
Keeping 'identity_test' as default agent
Bedrock AgentCore configured: /Users/sanghwa/workspaces/sketch/agentcore-identity-runtime-propagation/.bedrock_agentcore.yaml


✓ Configured WITH Authorization header forwarding


In [42]:
print("Deploying agent...\n")
launch_result = agentcore_runtime.launch()
print(f"\n✓ Deployed: {launch_result.agent_arn}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Creating memory resource for agent: identity_test
✅ MemoryManager initialized for region: us-west-2


Deploying agent...



🔎 Retrieving memory resource with ID: identity_test_mem-gx6EYnC8r5...
  Found memory: identity_test_mem-gx6EYnC8r5
Found existing memory in cloud: identity_test_mem-gx6EYnC8r5
Existing memory has 0 strategies
✅ Using existing STM-only memory
Starting CodeBuild ARM64 deployment for agent 'identity_test' to account 269550163595 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: identity_test
✅ ECR repository available: 269550163595.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-identity_test
Getting or creating execution role for agent: identity_test
Using AWS region: us-west-2, account ID: 269550163595
Role name: AmazonBedrockAgentCoreSDKRuntime-us-west-2-5f0c170d0d


✅ Reusing existing ECR repository: 269550163595.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-identity_test


✅ Reusing existing execution role: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-5f0c170d0d
✅ Execution role available: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-5f0c170d0d
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: identity_test
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-5f0c170d0d
Reusing existing CodeBuild execution role: arn:aws:iam::269550163595:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-5f0c170d0d
Using dockerignore.template with 45 patterns for zip filtering
Uploaded source to S3: identity_test/source.zip
Updated CodeBuild project: bedrock-agentcore-identity_test-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.5s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOUR


✓ Deployed: arn:aws:bedrock-agentcore:us-west-2:269550163595:runtime/identity_test-xZApwJAqnL


In [34]:
print("Waiting for READY status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']

while status not in ['READY', 'CREATE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"  {status}")

if status == 'READY':
    print(f"\n✓ Agent is {status}")
else:
    raise Exception(f"Deployment failed: {status}")

Waiting for READY status...


✅ MemoryManager initialized for region: us-west-2
🔎 Retrieving memory resource with ID: identity_test_mem-gx6EYnC8r5...
  Found memory: identity_test_mem-gx6EYnC8r5
Retrieved Bedrock AgentCore status for: identity_test



✓ Agent is READY


## Step 4: Test Identity Propagation

In [35]:
def get_fresh_tokens(username, password):
    """Get fresh tokens right before testing"""
    response = cognito_client.initiate_auth(
        ClientId=cognito_config['client_id'],
        AuthFlow='USER_PASSWORD_AUTH',
        AuthParameters={'USERNAME': username, 'PASSWORD': password}
    )
    return response['AuthenticationResult']

print("Authenticating users (tokens valid for 1 hour)...")
user_tokens = {}

for user in test_users:
    tokens = get_fresh_tokens(user['username'], user['password'])
    user_tokens[user['username']] = tokens
    print(f"  ✓ {user['username']}")

Authenticating users (tokens valid for 1 hour)...
  ✓ user-a
  ✓ user-b


### Token Analysis: Access Token vs ID Token

In [36]:
def decode_and_show_token(token, token_type):
    parts = token.split('.')
    payload = parts[1]
    padding = 4 - (len(payload) % 4)
    if padding != 4:
        payload += '=' * padding
    claims = json.loads(base64.b64decode(payload))
    
    print(f"\n{token_type}:")
    print(f"  aud: {claims.get('aud', 'NOT FOUND')}")
    print(f"  token_use: {claims.get('token_use')}")
    print(f"  sub: {claims.get('sub')[:20]}...")
    print(f"  username: {claims.get('username') or claims.get('cognito:username', 'NOT FOUND')}")
    print(f"  custom:tenant_id: {claims.get('custom:tenant_id', 'NOT FOUND')}")
    print(f"  custom:user_role: {claims.get('custom:user_role', 'NOT FOUND')}")
    return claims

print("=" * 60)
print("TOKEN ANALYSIS")
print("=" * 60)

user_a_tokens = user_tokens['user-a']
access_claims = decode_and_show_token(user_a_tokens['AccessToken'], 'Access Token')
id_claims = decode_and_show_token(user_a_tokens['IdToken'], 'ID Token')

print("\n" + "=" * 60)
print("CONCLUSION:")
print("  Access Token: aud missing → may fail allowedAudience check")
print("  ID Token: aud present + custom attributes → USE THIS")
print("=" * 60)

TOKEN ANALYSIS

Access Token:
  aud: NOT FOUND
  token_use: access
  sub: 6821b320-60a1-70a5-8...
  username: user-a
  custom:tenant_id: NOT FOUND
  custom:user_role: NOT FOUND

ID Token:
  aud: 5gmm9nquhjt9df0rok9fvkdb77
  token_use: id
  sub: 6821b320-60a1-70a5-8...
  username: user-a
  custom:tenant_id: tenant-a
  custom:user_role: admin

CONCLUSION:
  Access Token: aud missing → may fail allowedAudience check
  ID Token: aud present + custom attributes → USE THIS


### Test 1: No Authentication (should fail)

In [30]:
print("\nTEST 1: No auth")
try:
    response = agentcore_runtime.invoke({"prompt": "test"})
    print("✗ UNEXPECTED: Succeeded without auth")
except Exception as e:
    if "AccessDeniedException" in str(e) or "403" in str(e):
        print("✓ EXPECTED: Auth required")
    else:
        print(f"✗ Unexpected: {e}")


TEST 1: No auth
✓ EXPECTED: Auth required


### Test 2: With ID Token

In [37]:
print("\nTEST 2: ID Token from user-a")
print("=" * 60)

try:
    # Get fresh token right before invoke
    fresh_tokens = get_fresh_tokens('user-a', 'TempPass123!')
    id_token = fresh_tokens['IdToken']
    
    response = agentcore_runtime.invoke(
        {"prompt": "What is 2+2?"},
        bearer_token=id_token
    )
    
    # Parse response
    resp_data = response.get('response')
    if isinstance(resp_data, str):
        try:
            parsed = json.loads(resp_data)
            agent_resp = parsed['response']
            identity = parsed['user_identity']
        except:
            agent_resp = resp_data
            identity = response.get('user_identity', {})
    else:
        agent_resp = resp_data
        identity = response.get('user_identity', {})
    
    print(f"Agent: {agent_resp}")
    print(f"\nIdentity:")
    print(f"  Authenticated: {identity.get('authenticated')}")
    print(f"  Tenant: {identity.get('tenant_id')}")
    print(f"  Role: {identity.get('user_role')}")
    print(f"  User: {identity.get('username')}")
    
    if identity.get('tenant_id') == 'tenant-a':
        print("\n✓ SUCCESS: Identity propagation working!")
    else:
        print("\n✗ FAILED: Identity not correct")
        
except Exception as e:
    print(f"\n✗ ERROR: {e}")
    print("\nTroubleshooting:")
    print("  1. Check if agent is READY")
    print("  2. Verify Cognito discovery URL is accessible")
    print("  3. Try reconfiguring with allowedAudience if needed")
    print(f"  4. Discovery URL: {cognito_config['discovery_url']}")


TEST 2: ID Token from user-a


✅ MemoryManager initialized for region: us-west-2
Memory is active, proceeding with invoke


Agent: 2+2 equals 4.

Identity:
  Authenticated: False
  Tenant: None
  Role: None
  User: None

✗ FAILED: Identity not correct


### Test 3: Multi-Tenant Isolation

In [38]:
print("\nTEST 3: Multi-Tenant Isolation")
print("=" * 60)

results = []

for user in test_users:
    try:
        fresh_tokens = get_fresh_tokens(user['username'], user['password'])
        id_token = fresh_tokens['IdToken']
        
        response = agentcore_runtime.invoke(
            {"prompt": "Hello"},
            bearer_token=id_token
        )
        
        resp_data = response.get('response')
        if isinstance(resp_data, str):
            try:
                parsed = json.loads(resp_data)
                identity = parsed['user_identity']
            except:
                identity = response.get('user_identity', {})
        else:
            identity = response.get('user_identity', {})
        
        results.append({
            'username': user['username'],
            'expected_tenant': user['tenant_id'],
            'actual_tenant': identity.get('tenant_id'),
            'role': identity.get('user_role')
        })
    except Exception as e:
        print(f"  ✗ {user['username']}: {e}")
        results.append({'username': user['username'], 'error': str(e)})

# Display results
print(f"\n{'User':<10} {'Expected':<12} {'Actual':<12} {'Role':<10} {'Status'}")
print("-" * 60)

all_success = True
for r in results:
    if 'error' in r:
        print(f"{r['username']:<10} {'ERROR'}")
        all_success = False
    else:
        match = "✓" if r['expected_tenant'] == r['actual_tenant'] else "✗"
        print(f"{r['username']:<10} {r['expected_tenant']:<12} {r['actual_tenant']:<12} {r['role']:<10} {match}")
        if r['expected_tenant'] != r['actual_tenant']:
            all_success = False

if all_success:
    print("\n✓ SUCCESS: Multi-tenant isolation verified!")
else:
    print("\n✗ FAILED: Issues detected")


TEST 3: Multi-Tenant Isolation

User       Expected     Actual       Role       Status
------------------------------------------------------------


TypeError: unsupported format string passed to NoneType.__format__

## Summary

In [14]:
print("\n" + "=" * 60)
print("IDENTITY PROPAGATION SUMMARY")
print("=" * 60)
print("\nArchitecture:")
print("  Client → Bearer Token (ID Token)")
print("         → AgentCore validates JWT")
print("         → Authorization header → BedrockAgentCoreContext")
print("         → Agent extracts claims")
print("\nKey Findings:")
print("  1. ID Token contains: tenant_id, user_role (custom attributes)")
print("  2. Access Token contains: sub, username (basic identity only)")
print("  3. JWT propagates through entire chain")
print("  4. BedrockAgentCoreContext.get_request_headers() provides access")
print("\nRecommendation: Use ID Token for multi-tenant authorization")
print("=" * 60)


IDENTITY PROPAGATION SUMMARY

Architecture:
  Client → Bearer Token (ID Token)
         → AgentCore validates JWT
         → Authorization header → BedrockAgentCoreContext
         → Agent extracts claims

Key Findings:
  1. ID Token contains: tenant_id, user_role (custom attributes)
  2. Access Token contains: sub, username (basic identity only)
  3. JWT propagates through entire chain
  4. BedrockAgentCoreContext.get_request_headers() provides access

Recommendation: Use ID Token for multi-tenant authorization


## Cleanup (Optional)

```python
# Uncomment to delete resources
# agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
# agentcore_control_client.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
# cognito_client.delete_user_pool(UserPoolId=cognito_config['user_pool_id'])
```